In [ ]:
import os

os.environ["HF_TOKEN"] = "SECRET_KEY"

# 1. Với dữ liệu Pháp Điển

In [3]:
import pandas as pd
import json
from src.preprocess_parquet import parse_article, parse_source_note, save



def preProcess(raw: pd.DataFrame, fix_path: str=None) -> pd.DataFrame :
    """
    Đọc thông tin từ dataset.
    Thực hiện extract metadata từ cột `source_note_text`.
    Thực hiện parse (phân nhỏ) nội dung các Điều `content_text`.
    """
    metadata = (raw["source_note_text"]
                .apply(parse_source_note)
                .apply(pd.Series))
    
    final = pd.concat(
        [raw, metadata],
        axis=1
    )

    final['content_text'] = final['content_text'].apply(parse_article)
    final['content_clause_count'] = final['content_text'].apply(lambda x: 
                                                                len(x.get('content', []))
                                                                )

    final = final[['legal_type', 'docs_code', 'docs_title', 'article_index', 'article_title', 'source_note_text', 'source_links', 'topic_title', 'subject_title', 'content_text', 'content_word_count', 'content_clause_count']]

    if fix_path is not None:
        fix_df = pd.read_csv(fix_path, index_col=0)
        fix_df["content_text"] = (
            fix_df["content_text"]
            .apply(json.loads)
        )
        final.update(fix_df)
        print(f"Update {len(fix_df)} samples thủ công.")

    return final

In [4]:
import pandas as pd

DROP_TOPIC = [
    'An ninh quốc gia',
    'Cán bộ, công chức, viên chức',
    'Quốc phòng',
    'Chính sách xã hội',
    'Thống kê',
    'Xây dựng pháp luật và thi hành pháp luật',
    'Tổ chức bộ máy nhà nước',
    'Ngoại giao, điều ước quốc tế',
    'Dân số, gia đình, trẻ em, bình đẳng giới',
    'Hình sự',
    'Tổ chức chính trị - xã hội, hội',
    'Tương trợ tư pháp',
    'Dân tộc',
    'Tôn giáo, tín ngưỡng',
    'Văn thư lưu trữ'
]

df = pd.read_parquet("data/raw/phapdien.parquet")

irrelevant_mask = df['topic_title'].isin(DROP_TOPIC)

df = df[~irrelevant_mask]
print(f"Drop {sum(irrelevant_mask)} samples theo chủ đề.")
    
empty_mask = df['content_text'] == ""
df = df[~empty_mask]
print(f"Drop {sum(empty_mask)} samples có nội dung trống.")

df = preProcess(df, fix_path = "data/raw/phapdien_error_correction.csv")

print(f"Còn lại tổng cộng {len(df)} samples.")
df.head(1)

Drop 5985 samples theo chủ đề.
Drop 117 samples có nội dung trống.
Update 8 samples thủ công.
Còn lại tổng cộng 41680 samples.


,legal_type,docs_code,docs_title,article_index,article_title,source_note_text,source_links,topic_title,subject_title,content_text,content_word_count,content_clause_count
947,Luật,25/2008/QH12,Luật số 25/2008/QH12,Điều 1,Điều 2.2.LQ.1. Phạm vi điều chỉnh và đối tượng...,(Điều 1 Luật số 25/2008/QH12 Bảo hiểm y tế ngà...,[{'text': '(Điều 1 Luật số 25/2008/QH12 Bảo hi...,Bảo hiểm,Bảo hiểm y tế,"{'title': '', 'content': [{'text': '1. Luật nà...",135,3


## Hậu kiểm với tmquan/vbpl-vn

Do dữ liệu của Pháp Điển có xen lẫn các tài liệu "Hết hiệu lực toàn bộ", nên ta cần lọc bỏ những trường hợp như thế đi.

Kho dữ liệu `tmquan/vbpl-vn` này chứa id của tài liệu trên `vbpl.vn`. Thông qua đó, ta có thể kiểm tra được status của tài liệu xem liệu còn hiệu lực hay không.

In [5]:
from datasets import load_dataset
import re
import numpy as np


def _normalize_doc_number(doc_numbers):
    """
    [" 3131/QĐ-UB-NCVX ", "12 / TT-BTC"]
    -> "3131/QĐ-UB-NCVX, 12/TT-BTC"
    """

    if doc_numbers is None:
        return None
    if not isinstance(doc_numbers, list) :
        doc_numbers = [doc_numbers]

    cleaned = []

    for x in doc_numbers:
        if x is None:
            continue

        # xóa toàn bộ khoảng trắng
        x = re.sub(r"\s+", "", str(x))

        if x:
            cleaned.append(x)

    return ", ".join(cleaned)

def format_vbpl_dataset(dataset):
    """
    Parameters
    ----------
    dataset : datasets.Dataset

    Returns
    -------
    pandas.DataFrame
    """

    rows = []

    for sample in dataset:
        try:
            rows.append({
                "id": sample.get("item_id", np.nan),
                "scope": sample.get("scope"),
                "subject_title": sample.get("legal_area"),
                "issue_date": sample.get("issue_date"),
                "source_url": sample.get("source_url"),
                "api_url": sample.get("api_url"),
                "legal_type": sample.get('legal_type'),
                "legal_area": sample.get('legal_area'),
                "doc_number": _normalize_doc_number(
                    sample.get("doc_number")
                ),
                "title": sample.get("title"),
            })
        except:
            pass

    return pd.DataFrame(rows)


In [6]:
ds = load_dataset("tmquan/vbpl-vn", split="train", token=os.environ["HF_TOKEN"])
df_vbpl = format_vbpl_dataset(ds)
df_vbpl.head(5)

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

,id,scope,subject_title,issue_date,source_url,api_url,legal_type,legal_area,doc_number,title
0,1,trung_uong,Chưa phân loại,1950-03-27,https://vbpl.vn/van-ban/chi-tiet/nghi-dinh-so-...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị định,Chưa phân loại,24/LĐ-NĐ,Tổ chức các cơ quan Lao động địa phương liên k...
1,10,trung_uong,Chưa phân loại,1950-10-16,https://vbpl.vn/van-ban/chi-tiet/thong-tu-so-4...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Thông tư,Chưa phân loại,41-NV-6-TT,Định thể lệ xếp công chức vào thang lương chun...
2,100,trung_uong,Chưa phân loại,1950-05-14,https://vbpl.vn/van-ban/chi-tiet/sac-lenh-so-6...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Sắc lệnh,Chưa phân loại,68/SL,thành lập Ban Kinh tế Chính phủ
3,1000,trung_uong,Chưa phân loại,1957-01-22,https://vbpl.vn/van-ban/chi-tiet/nghi-quyet-so...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị quyết,Chưa phân loại,Khôngsố,Về việc hoàn toàn tín nhiệm Chính phủ
4,10000,trung_uong,Chưa phân loại,1995-01-25,https://vbpl.vn/van-ban/chi-tiet/chi-thi-so-64...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Chỉ thị,Chưa phân loại,64-TTg,"Về tăng cường công tác giải quyết khiếu nại, t..."


Khi có đủ dữ liệu từ `vbpl-vn` thì ta sẽ đối chiếu theo doc_number để truy ra các tài liệu có trùng lặp giữa 2 bộ

In [24]:
code_list = df['docs_code'].unique().tolist()

# 2. Lọc các dòng có 'doc_number' nằm trong danh sách trên
df_left = df_vbpl[df_vbpl['doc_number'].isin(code_list)]

print(f"Số lượng docs trùng lặp: {len(df_left)}")

Số lượng docs trùng lặp: 3255


In [25]:
phapdien_unique_df = df[~df['docs_code'].isin(df_left['doc_number'].unique().tolist())].sort_values('content_word_count', ascending=False)

save(phapdien_unique_df, "data/processed/phapdien_processed_unique.parquet")

print(f'Số lượng tài liệu chỉ tìm thấy ở Pháp Điển: {len(phapdien_unique_df)}/{len(df)}')
phapdien_unique_df.head(2)

Đã lưu dữ liệu vào data/processed/phapdien_processed_unique.parquet
Số lượng tài liệu chỉ tìm thấy ở Pháp Điển: 1476/41680


,legal_type,docs_code,docs_title,article_index,article_title,source_note_text,source_links,topic_title,subject_title,content_text,content_word_count,content_clause_count
47500,Quyết định,47/2013/QĐ-TTG,Quyết định số 47/2013/QĐ-TTg,Điều 2,Điều 45.9.QĐ.1.2. Điều lệ tổ chức và hoạt động...,"(Điều 2 Quyết định số 47/2013/QĐ-TTg, có hiệu ...",[{'text': '(Điều 2 Quyết định số 47/2013/QĐ-TT...,"Y tế, dược","Phòng, chống tác hại của thuốc lá",{'title': 'Phê duyệt và ban hành kèm theo Quyế...,6224,26
29077,Thông tư,23/2023/TT-BTC,Thông tư số 23/2023/TT-BTC,Điều 6,Điều 28.3.TT.28.6. Xác định nguyên giá tài sản...,"(Điều 6 Thông tư số 23/2023/TT-BTC, có hiệu lự...","[{'text': '(Điều 6 Thông tư số 23/2023/TT-BTC,...","Tài sản công, nợ công, dự trữ nhà nước","Quản lý, sử dụng tài sản công",{'title': 'Việc xác định nguyên giá tài sản cố...,4376,5


In [ ]:
from bs4 import BeautifulSoup
import requests

FAIL_INDICES = []
DOWNLOADED = []
REQUEST_ERROR = []
# Biến đếm để hiển thị trên thanh tqdm
success_count = 0
article_count = 0


MAX_WORKERS = 32

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/xml, text/xml, */*; q=0.01'
}

def parse_document(soup: BeautifulSoup) -> dict:
    
    # Kiểm tra xem thẻ <data> có tồn tại không
    data_tag = soup.find("data")
    if not data_tag:
        return None

    # 1. Trích xuất các trường từ gốc <data>
    # Dùng find(recursive=False) để chỉ lấy thẻ con trực tiếp, tránh trùng với id của documentContent/documentIssues...
    doc_id = (
        data_tag.find("id", recursive=False).text
        if data_tag.find("id", recursive=False)
        else None
    )
    docs_code = (
        data_tag.find("docNum").text if data_tag.find("docNum") else None
    )
    article_title = (
        data_tag.find("title").text if data_tag.find("title") else None
    )
    issue_date = (
        data_tag.find("issueDate").text if data_tag.find("issueDate") else None
    )
    eff_from = (
        data_tag.find("effFrom").text if data_tag.find("effFrom") else None
    )
    eff_to = data_tag.find("effTo").text if data_tag.find("effTo") else None
    agency_name = (
        data_tag.find("agencyName").text if data_tag.find("agencyName") else None
    )

    # Lấy status từ effStatus -> name
    eff_status_tag = data_tag.find("effStatus")
    status = (
        eff_status_tag.find("name").text
        if eff_status_tag and eff_status_tag.find("name")
        else None
    )


    # 3. Trường documentMajors -> name (topic_title)
    doc_majors_tag = data_tag.find("documentMajors")
    topic_title = (
        doc_majors_tag.find("name").text
        if doc_majors_tag and doc_majors_tag.find("name")
        else None
    )

    # 4. Trường documentFields -> name (subject_title)
    doc_fields_tag = data_tag.find("documentFields")
    subject_title = (
        doc_fields_tag.find("name").text
        if doc_fields_tag and doc_fields_tag.find("name")
        else None
    )


    # Tổng hợp thành dictionary
    result_dict = {
        "id": doc_id,
        "docs_code": _normalize_doc_number(docs_code),
        "article_title": article_title,
        "issueDate": issue_date,
        "effFrom": eff_from,
        "effTo": eff_to,
        "status": status,
        "agency_name": agency_name,
        "topic_title": topic_title,
        "subject_title": subject_title,
    }

    return result_dict

def trecking_work(row:pd.Series):
    item_id = str(row["id"])
    api_url = str(row['api_url'])
    scope = str(row['scope'])

    # print(item_id)
    
    if not api_url or api_url == 'nan':
        if not item_id :
            FAIL_INDICES.append(row.name)
        else :
            api_url = f'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/{item_id}'

    try:
        # Timeout 10s để tránh treo vĩnh viễn
        res = requests.get(api_url, headers=HEADERS, timeout=30) 
        res.raise_for_status()
        
        xml_soup = BeautifulSoup(res.content, 'xml')

        results = parse_document(xml_soup)

        results['scope'] = scope

        return results
    except requests.exceptions.Timeout:
        FAIL_INDICES.append(row.name)
        # print(f"Timout at {row.name}")
        return None
    except requests.exceptions.HTTPError as http_err:
        # Xử lý các lỗi HTTP không OK (ví dụ: 400, 401, 403, 404, 500...) trừ mã 304 đã bắt ở trên
        print(f"[ERROR] Lỗi HTTP xảy ra: loc={row.name} (Status code: {res.status_code})")
        REQUEST_ERROR.append(row)
        return None
    except Exception:
        FAIL_INDICES.append(row.name)
        # print(f"Error at {row.name}")
        return None

In [ ]:

df_left.loc[df_left['doc_number'] == '21/2015/NĐ-CP', 'api_url'] = 'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/52378'

REQUEST_ERROR = []

In [ ]:
from concurrent import futures as thread_futures
from tqdm.notebook import tqdm



for idx in range(3) :
    if len(df_left) == 0 : break

    FAIL_INDICES = []

    with thread_futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 2. Truyền row (kiểu pd.Series) vào hàm. Lúc này row.name sẽ là chỉ số iloc
        futures = [
            executor.submit(trecking_work, row) for _, row in df_left.iterrows()
        ]

        # 3. Tạo thanh tiến trình với cấu hình postfix ban đầu
        pbar = tqdm(
            thread_futures.as_completed(futures), total=len(futures), desc=f"Crawling {idx}"
        )

        for future in pbar:
            result = future.result()

            if result is not None:
                success_count += 1
                DOWNLOADED.append(result)

            # 4. Cập nhật postfix theo thời gian thực
            pbar.set_postfix(OK=success_count, Redo=len(FAIL_INDICES), Error=len(REQUEST_ERROR))

    # 5. Chuyển list dict thành DataFrame hoàn chỉnh
    status_df = pd.DataFrame(DOWNLOADED)
    df_left   = df_vbpl.loc[FAIL_INDICES]

df_left = pd.concat([df_left, pd.DataFrame(REQUEST_ERROR)], ignore_index=False)

Crawling 0:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
df_left

,id,scope,subject_title,issue_date,source_url,api_url,legal_type,legal_area,doc_number,title
76302,180733,trung_uong,Chưa phân loại,2001-06-01,https://vbpl.vn/van-ban/chi-tiet/nghi-dinh-so-...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị định,Chưa phân loại,21/2015/NĐ-CP,"quy định về nhuận bút, thù lao đối với tác phẩ..."


In [38]:
out_of_date = status_df[
    (status_df['status'].isin(['Hết hiệu lực toàn bộ', 'Không còn phù hợp', 'Ngưng hiệu lực']))
    | (status_df['scope'] == 'dia_phuong')    
]['docs_code'].unique().tolist()

final_df = df[~df['docs_code'].isin(out_of_date)]


In [39]:
print(f"Số lượng còn lại sau khi lọc (hết hiệu lực, vbpl địa phương): {len(final_df)}/{len(df)}")

final_df = final_df.sort_values('content_word_count', ascending=False)
final_df

Số lượng còn lại sau khi lọc (hết hiệu lực, vbpl địa phương): 34279/41680


,legal_type,docs_code,docs_title,article_index,article_title,source_note_text,source_links,topic_title,subject_title,content_text,content_word_count,content_clause_count
40313,Thông tư,18/2023/TT-BTC,Thông tư số 18/2023/TT-BTC,Điều 6,Điều 39.13.TT.70.6. Cách tính và thực hiện bù ...,"(Điều 6 Thông tư số 18/2023/TT-BTC, có hiệu lự...","[{'text': '(Điều 6 Thông tư số 18/2023/TT-BTC,...","Trật tự, an toàn xã hội",Xử lý vi phạm hành chính,"{'title': '', 'content': [{'text': '1. Cách tí...",1735769,63
27764,Nghị định,38/2016/NĐ-CP,Nghị định số 38/2016/NĐ-CP,Điều 8,Điều 27.2.NĐ.1.8. Bảo vệ hành lang kỹ thuật cô...,"(Điều 8 Nghị định số 38/2016/NĐ-CP, có hiệu lự...","[{'text': '(Điều 8 Nghị định số 38/2016/NĐ-CP,...",Tài nguyên,Khí tượng thủy văn,"{'title': '', 'content': [{'title': '1. Bảo vệ...",116725,22
22725,Thông tư,36/2014/TT-BNNPTNT,Thông tư số 36/2014/TT-BNNPTNT,Điều 2,Điều 24.1.TT.12.2. Giải thích từ ngữ,"(Điều 2 Thông tư số 36/2014/TT-BNNPTNT, có hiệ...",[{'text': '(Điều 2 Thông tư số 36/2014/TT-BNNP...,"Nông nghiệp, nông thôn",Bảo vệ và kiểm dịch thực vật,"{'title': 'Trong Thông tư này, các từ ngữ dưới...",95736,30
18195,Nghị định,145/2020/NĐ-CP,Nghị định số 145/2020/NĐ-CP,Điều 57,Điều 20.2.NĐ.3.57. Tiền lương làm thêm giờ vào...,"(Điều 57 Nghị định số 145/2020/NĐ-CP, có hiệu ...",[{'text': '(Điều 57 Nghị định số 145/2020/NĐ-C...,Lao động,Lao động,{'title': 'Người lao động làm thêm giờ vào ban...,89094,14
2096,Thông tư,20/2017/TT-BTTTT,Thông tư số 20/2017/TT-BTTTT,Điều 3,Điều 3.1.TT.3.3. Phân cấp tổ chức thực hiện ứn...,"(Điều 3 Thông tư số 20/2017/TT-BTTTT, có hiệu ...",[{'text': '(Điều 3 Thông tư số 20/2017/TT-BTTT...,"Bưu chính, viễn thông",An toàn thông tin mạng,{'title': 'Phân cấp tổ chức thực hiện ứng cứu ...,64554,14
...,...,...,...,...,...,...,...,...,...,...,...,...
27708,Thông tư,21/2016/TT-BTNMT,Thông tư số 21/2016/TT-BTNMT,Điều 1,Điều 27.1.TT.53.1. Ban hành kèm theo Thông tư ...,(Điều 1 Thông tư số 21/2016/TT-BTNMT Ban hành ...,[{'text': '(Điều 1 Thông tư số 21/2016/TT-BTNM...,Tài nguyên,Đo đạc và bản đồ,{'text': 'Danh muc_21_2016_TT-BTNMT.doc'},2,0
27716,Thông tư,19/2017/TT-BTNMT,Thông tư số 19/2017/TT-BTNMT,Điều 1,Điều 27.1.TT.62.1. Ban hành kèm theo Thông tư ...,(Điều 1 Thông tư số 19/2017/TT-BTNMT Ban hành ...,[{'text': '(Điều 1 Thông tư số 19/2017/TT-BTNM...,Tài nguyên,Đo đạc và bản đồ,{'text': 'Danh muc_19_2017_TT-BTNMT.doc'},2,0
27718,Thông tư,48/2013/TT-BTNMT,Thông tư số 48/2013/TT-BTNMT,Điều 1,Điều 27.1.TT.26.1. Ban hành kèm theo Thông tư ...,(Điều 1 Thông tư số 48/2013/TT-BTNMT Ban hành ...,[{'text': '(Điều 1 Thông tư số 48/2013/TT-BTNM...,Tài nguyên,Đo đạc và bản đồ,{'text': 'Danh muc_48_2013_TT-BTNMT.doc'},2,0
27691,Thông tư,38/2013/TT-BTNMT,Thông tư số 38/2013/TT-BTNMT,Điều 1,Điều 27.1.TT.21.1. Ban hành kèm theo Thông tư ...,(Điều 1 Thông tư số 38/2013/TT-BTNMT Ban hành ...,[{'text': '(Điều 1 Thông tư số 38/2013/TT-BTNM...,Tài nguyên,Đo đạc và bản đồ,{'text': 'Danh muc_38_2013_TT-BTNMT.doc'},2,0


## Lưu dữ liệu Pháp Điển đã xử lý

In [40]:
save(final_df, "data/processed/phapdien_processed.parquet")

Đã lưu dữ liệu vào data/processed/phapdien_processed.parquet
